# Weather-Based Energy Consumption Prediction
This notebook covers the complete Machine Learning pipeline for predicting hourly electricity consumption using weather features. The dataset is a merged version of the American Electric Power (AEP) hourly energy consumption and historical weather variables for Columbus, Ohio (October 2004 – August 2018).

## Objectives:
1. **Data Cleaning & Preprocessing**: Handle missing values, duplicate entries, and formats.
2. **Exploratory Data Analysis (EDA)**: Visualize the relationship between weather factors and energy consumption.
3. **Feature Engineering**: Extract temporal components and encode categorical weather conditions.
4. **Model Training & Comparison**: Train Linear Regression, Decision Tree, Random Forest, and Gradient Boosting models.
5. **Model Evaluation & Serialization**: Evaluate models using MAE, RMSE, and $R^2$, then save the best performing model.

### Step 1: Import Libraries and Load Dataset

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set style for plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Load dataset
df = pd.read_csv("../dataset/energy_weather.csv")
print(f"Dataset Loaded. Shape: {df.shape}")
print(df.head())

### Step 2: Data Cleaning

In [ ]:
# Check for missing values
print("Missing Values:\n", df.isnull().sum())

# Check for duplicate entries
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates dropped.")

### Step 3: Feature Engineering
We parse the `Datetime` column to extract key temporal features: `Hour`, `Day`, `Month`, `Year`, and `Weekday`. We also encode `Weather Condition` (Sunny, Cloudy, Rainy) into numerical values.

In [ ]:
# Convert Datetime
df['Datetime'] = pd.to_datetime(df['Datetime'], format='%d-%m-%Y %H:%M')

# Extract features
df['Hour'] = df['Datetime'].dt.hour
df['Month'] = df['Datetime'].dt.month
df['Day'] = df['Datetime'].dt.day
df['Weekday'] = df['Datetime'].dt.weekday
df['Year'] = df['Datetime'].dt.year

# Encode Weather Condition
weather_map = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2}
df['Weather_Encoded'] = df['Weather Condition'].map(weather_map)

print("Features engineered successfully!")
print(df[['Datetime', 'Hour', 'Month', 'Weekday', 'Weather Condition', 'Weather_Encoded']].head())

### Step 4: Exploratory Data Analysis (EDA)
We will explore the dataset through several visualizations to identify correlations and seasonal/hourly patterns.

In [ ]:
# 1. Temperature vs Energy Consumption
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(2000, random_state=42), x='Temperature', y='AEP_MW', alpha=0.5, color='coral')
plt.title('Temperature vs. Energy Consumption (Sample of 2000 points)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Energy Consumption (MW)')
plt.savefig('../dataset/temp_vs_energy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Humidity vs Energy Consumption
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(2000, random_state=42), x='Humidity', y='AEP_MW', alpha=0.5, color='skyblue')
plt.title('Humidity vs. Energy Consumption (Sample of 2000 points)')
plt.xlabel('Humidity (%)')
plt.ylabel('Energy Consumption (MW)')
plt.savefig('../dataset/humidity_vs_energy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Monthly Consumption Patterns
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Month', y='AEP_MW', hue='Month', palette='tab20', legend=False)
plt.title('Energy Consumption Patterns by Month')
plt.xlabel('Month')
plt.ylabel('Energy Consumption (MW)')
plt.savefig('../dataset/monthly_consumption.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Hourly Consumption Patterns
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Hour', y='AEP_MW', hue='Weather Condition', errorbar=None, linewidth=2.5)
plt.title('Average Hourly Energy Consumption by Weather Condition')
plt.xlabel('Hour of the Day')
plt.ylabel('Energy Consumption (MW)')
plt.savefig('../dataset/hourly_consumption.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Correlation Heatmap
plt.figure(figsize=(10, 8))
num_cols = ['AEP_MW', 'Temperature', 'Humidity', 'Wind Speed', 'Pressure', 
            'Cloud Cover', 'Weather_Encoded', 'Hour', 'Weekday', 'Month', 'Year']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.savefig('../dataset/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

### Step 5: Model Selection and Training

In [ ]:
# Define features and target
features = [
    'Temperature', 'Humidity', 'Wind Speed', 'Pressure', 
    'Cloud Cover', 'Weather_Encoded', 'Hour', 'Weekday', 'Month', 'Year'
]
target = 'AEP_MW'

X = df[features]
y = df[target]

# Split the data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Train baseline and regression models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
}

results = {}
best_r2 = -1
best_model_name = None
best_model = None

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    
    results[name] = {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}
    print(f"{name} - MAE: {mae:.2f} MW, RMSE: {rmse:.2f} MW, R2: {r2:.4f}\n")
    
    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name
        best_model = model

### Step 6: Save the Best Model
We save the best performing model (highest $R^2$ score) along with metadata using `pickle`.

In [ ]:
print(f"Saving best model: {best_model_name} with R2: {best_r2:.4f}")

model_path = "../model/energy_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump({
        'model': best_model,
        'features': features,
        'weather_map': weather_map,
        'metrics': results[best_model_name],
        'all_metrics': results
    }, f)
print("Model saved successfully!")